In [6]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
import statsmodels.formula.api as smf
from scipy.special import expit, logit
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [7]:
warnings.filterwarnings('ignore')
df = pd.read_csv('../Data_Clean/data/data_cleaned/Weight_Stage1_Cleaned_Panel.csv')
future_years = list(range(2026, 2036))
lads = df['ONS_code'].unique()

skeleton = pd.DataFrame([(lad, yr) for lad in lads for yr in future_years], columns=['ONS_code', 'year'])
df_full = pd.concat([df, skeleton], ignore_index=True)

# 补齐所有静态地理字段，避免后续出现 NaN
geo_map = df[['ONS_code', 'ONS_geo', 'Geo_level', 'Country', 'Region']].dropna().drop_duplicates().set_index('ONS_code')
for col in geo_map.columns:
    df_full[col] = df_full['ONS_code'].map(geo_map[col])

df_full = df_full.sort_values(by=['ONS_code', 'year']).reset_index(drop=True)


for lad in lads:
    mask_lad = df_full['ONS_code'] == lad
    nation = df_full.loc[mask_lad, 'Country'].iloc[0]
    train_yr_pop = 2024 if nation in ['England', 'Wales'] else 2023

    # 拟合人口
    df_train_pop = df_full[mask_lad & (df_full['year'] <= train_yr_pop)]
    if len(df_train_pop) > 2:
        slope_p, intcpt_p = np.polyfit(df_train_pop['year'], np.log(df_train_pop['pop_value']), 1)
        pred_yrs_pop = df_full.loc[mask_lad & (df_full['year'] > train_yr_pop), 'year']
        df_full.loc[pred_yrs_pop.index, 'pop_value'] = np.exp(slope_p * pred_yrs_pop + intcpt_p).values

    # 拟合财富 (假设用2011-2023推演未来)
    df_train_gdhi = df_full[mask_lad & (df_full['year'] <= 2023)]
    if len(df_train_gdhi) > 2:
        slope_g, intcpt_g = np.polyfit(df_train_gdhi['year'], np.log(df_train_gdhi['gdhi_per_head']), 1)
        pred_yrs_gdhi = df_full.loc[mask_lad & (df_full['year'] >= 2024), 'year']
        df_full.loc[pred_yrs_gdhi.index, 'gdhi_per_head'] = np.exp(slope_g * pred_yrs_gdhi + intcpt_g).values

    # 拟合车辆 (基于历史人车比与 ±5% 漂移)
    df_train_veh = df_full[mask_lad & (df_full['year'] >= 2021) & (df_full['year'] <= 2023)]

    if len(df_train_veh) > 1:
        # 提取历史人车比
        hist_ratios = df_train_veh['veh_total'] / df_train_veh['pop_value']
        ratio_2023 = hist_ratios.iloc[-1]

        # 计算 CAGR 漂移率(±0.5% 每年)
        ratio_cagr = (hist_ratios.iloc[-1] / hist_ratios.iloc[0])**(1/(len(hist_ratios)-1)) - 1
        ratio_cagr_capped = np.clip(ratio_cagr, -0.005, 0.005)

        # 预测未来年份的人车比，并乘以未来人口得到车辆总数
        pred_yrs_veh = df_full.loc[mask_lad & (df_full['year'] >= 2024), 'year']
        future_ratios = ratio_2023 * (1 + ratio_cagr_capped)**(pred_yrs_veh.values - 2023)
        future_pop = df_full.loc[pred_yrs_veh.index, 'pop_value'].values

        df_full.loc[pred_yrs_veh.index, 'veh_total'] = future_pop * future_ratios

def logistic_growth(t, r, t0, K):
    return K / (1 + np.exp(-r * (t - t0)))

for lad in lads:
    mask_lad = df_full['ONS_code'] == lad
    df_train_ports = df_full[mask_lad & (df_full['year'] >= 2019) & (df_full['year'] <= 2025)]

    # 基于 CMA 和 EAFO 的公式推导 K_limit
    veh_2035 = df_full.loc[mask_lad & (df_full['year'] == 2035), 'veh_total'].values[0]
    K_limit = veh_2035 / 40.0

    years_hist = df_train_ports['year'].values
    ports_hist = df_train_ports['charging_ports'].values

    if len(ports_hist) > 3 and ports_hist[-1] > ports_hist[0]:
        try:
            popt, _ = curve_fit(lambda t, r, t0: logistic_growth(t, r, t0, K_limit),
                                years_hist, ports_hist,
                                p0=[0.4, 2027],
                                bounds=([0.01, 2020], [1.5, 2040]))

            r_fit = popt[0]
            t0_fit = popt[1]

            pred_yrs_ports = df_full.loc[mask_lad & (df_full['year'] >= 2026), 'year']
            pred_ports = logistic_growth(pred_yrs_ports.values, r_fit, t0_fit, K_limit)
            df_full.loc[pred_yrs_ports.index, 'charging_ports'] = pred_ports

        except RuntimeError:
            slope_fallback = (ports_hist[-1] - ports_hist[0]) / (years_hist[-1] - years_hist[0])
            pred_yrs_ports = df_full.loc[mask_lad & (df_full['year'] >= 2026), 'year']
            df_full.loc[pred_yrs_ports.index, 'charging_ports'] = ports_hist[-1] + slope_fallback * (pred_yrs_ports.values - 2025)


df_full['ports_per_10k_pop'] = (df_full['charging_ports'] / df_full['pop_value']) * 100000
df_full['charging_ports'] = df_full['charging_ports'].round(0)

# 时序排序
df_full = df_full.sort_values(by=['ONS_code', 'year']).reset_index(drop=True)

# 基于新数据的滞后列序列
new_lag_ports = df_full.groupby('ONS_code')['ports_per_10k_pop'].shift(1)
new_lag_gdhi = df_full.groupby('ONS_code')['gdhi_per_head'].shift(1)
new_lag_cp = df_full.groupby('ONS_code')['charging_ports'].shift(1)

# 确保 2019 年里的历史 Lag 记忆不被擦除
mask = df_full['year'] >= 2020
df_full.loc[mask, 'lag1_ports_per_10k_pop'] = new_lag_ports[mask]
df_full.loc[mask, 'lag1_gdhi_per_head'] = new_lag_gdhi[mask]
df_full.loc[mask, 'lag1_charging_ports'] = new_lag_cp[mask]

df_full['ports_per_10k_pop'] = df_full['ports_per_10k_pop'].round(1)
df_full['lag1_ports_per_10k_pop'] = df_full['lag1_ports_per_10k_pop'].round(1)
df_full['gdhi_per_head'] = df_full['gdhi_per_head'].round(1)
df_full['lag1_gdhi_per_head'] = df_full['lag1_gdhi_per_head'].round(1)
df_full['pop_value'] = df_full['pop_value'].round(0)
df_full['veh_total'] = df_full['veh_total'].round(0)
# 导出
df_full.to_csv('Stage4_Module1_Future_Features.csv', index=False)
print("未来自变量准备完毕。")

未来自变量准备完毕。


In [8]:
# 1. 加载底表
df = pd.read_csv('Stage4_Module1_Future_Features.csv')
df = df.sort_values(by=['ONS_code', 'year']).reset_index(drop=True)

df_train = df[(df['year'] >= 2020) & (df['year'] <= 2025)].copy()
df_train = df_train.dropna(subset=['lag1_ports_per_10k_pop', 'lag1_gdhi_per_head', 'ev_penetration', 'pop_value'])

# 基于英国《2023年车辆排放交易机制令》推导的极限存量更替率
FLEET_CEILING = 0.5024
P_actual = (df_train['ev_penetration'] / 100.0).clip(0.0001, FLEET_CEILING - 0.001)
df_train['logit_ev'] = logit(P_actual / FLEET_CEILING)

# 2. 拟合带有 LAD 固定效应的面板线性模型
formula = "logit_ev ~ year + np.log1p(lag1_ports_per_10k_pop) + np.log1p(lag1_gdhi_per_head) + np.log1p(pop_value) + C(ONS_code)"

model_prediction = smf.ols(formula=formula, data=df_train).fit()

# 3. 外推
df_future = df[df['year'] >= 2026].copy()

# 预测
predicted_logits = model_prediction.predict(df_future)

# 升维还原：公式 P = K / (1 + e^-Y)
# 这里乘回 100 是为了恢复成原表的百分号显示格式
df_future['ev_penetration'] = FLEET_CEILING * expit(predicted_logits) * 100

# 4. 衍生物理指标
df_future['veh_ulev_total'] = (df_future['ev_penetration'] / 100.0) * df_future['veh_total']
df_future['veh_ulev_total'] = df_future['veh_ulev_total'].round(0)

df_future['ulev_to_port_ratio'] = df_future['veh_ulev_total'] / df_future['charging_ports']
df_future['ulev_to_port_ratio'] = df_future['ulev_to_port_ratio'].round(2)

# 5. 导出
df_final = pd.concat([
    df[df['year'] <= 2025],
    df_future
], ignore_index=True).sort_values(by=['ONS_code', 'year'])

df_final['ev_penetration'] = df_final['ev_penetration'].round(2)
output_filename = 'Stage4_Module2_Final_EV_Predictions_2035.csv'
df_final.to_csv(output_filename, index=False)

print(f"2035渗透率与绝对车辆盘已预测完毕，结果保存至 Stage4_Module2_Final_EV_Predictions_2035.csv")

2035渗透率与绝对车辆盘已预测完毕，结果保存至 Stage4_Module2_Final_EV_Predictions_2035.csv


In [9]:

# 1. 加载数据
df = pd.read_csv('Stage4_Module1_Future_Features.csv')
df = df.sort_values(by=['ONS_code', 'year']).reset_index(drop=True)

df_train = df[(df['year'] >= 2020) & (df['year'] <= 2025)].copy()
df_train = df_train.dropna(subset=['lag1_ports_per_10k_pop', 'lag1_gdhi_per_head', 'ev_penetration', 'pop_value'])

FLEET_CEILING = 0.5024
P_actual = (df_train['ev_penetration'] / 100.0).clip(0.0001, FLEET_CEILING - 0.001)
df_train['logit_ev'] = logit(P_actual / FLEET_CEILING)

# 重构模型
formula = "logit_ev ~ year + np.log1p(lag1_ports_per_10k_pop) + np.log1p(lag1_gdhi_per_head) + np.log1p(pop_value) + C(ONS_code)"
model = smf.ols(formula=formula, data=df_train).fit()

# 提取未来 95% 误差带
df_future = df[df['year'] >= 2026].copy()
pred_results = model.get_prediction(df_future)
pi = pred_results.conf_int(obs=True, alpha=0.05)

df_future['ev_mean'] = FLEET_CEILING * expit(pred_results.predicted_mean) * 100
df_future['ev_lower'] = FLEET_CEILING * expit(pi[:, 0]) * 100
df_future['ev_upper'] = FLEET_CEILING * expit(pi[:, 1]) * 100

# 计算绝对物理车辆
df_future['veh_mean'] = (df_future['ev_mean'] / 100.0) * df_future['veh_total']
df_future['veh_lower'] = (df_future['ev_lower'] / 100.0) * df_future['veh_total']
df_future['veh_upper'] = (df_future['ev_upper'] / 100.0) * df_future['veh_total']

df_hist = df[df['year'] <= 2025].copy()
df_hist['veh_mean'] = (df_hist['ev_penetration'] / 100.0) * df_hist['veh_total']
df_hist['veh_lower'] = df_hist['veh_mean']
df_hist['veh_upper'] = df_hist['veh_mean']

cols = ['Country', 'Region', 'year', 'veh_total', 'veh_mean', 'veh_lower', 'veh_upper']
df_plot = pd.concat([df_hist[cols], df_future[cols]], ignore_index=True)

# 2. 除权聚合
country_df = df_plot.groupby(['Country', 'year'])[['veh_mean', 'veh_lower', 'veh_upper', 'veh_total']].sum().reset_index()
country_df['reg_mean'] = (country_df['veh_mean'] / country_df['veh_total']) * 100
country_df['reg_lower'] = (country_df['veh_lower'] / country_df['veh_total']) * 100
country_df['reg_upper'] = (country_df['veh_upper'] / country_df['veh_total']) * 100

region_df = df_plot.groupby(['Region', 'year'])[['veh_mean', 'veh_total']].sum().reset_index()
region_df['reg_mean'] = (region_df['veh_mean'] / region_df['veh_total']) * 100


sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

# 3. 绘制四国单线分段 S 曲线
countries = country_df['Country'].unique()
for country in countries:
    data = country_df[country_df['Country'] == country]

    hist_data = data[data['year'] <= 2025]
    pred_data = data[data['year'] >= 2025]

    fig, ax = plt.subplots(figsize=(10, 6))

    # 真实数据段 (蓝色实线 + 实心点)
    ax.plot(hist_data['year'], hist_data['reg_mean'], color='#1f77b4', linewidth=2.5, marker='o', markersize=6, label='Historical Data', zorder=4)

    # 预测数据段 (橙色虚线 + 实心点)
    ax.plot(pred_data['year'], pred_data['reg_mean'], color='#ff7f0e', linewidth=2.5, linestyle='--', marker='o', markersize=6, label='Predicted Data', zorder=4)

    # 95% 置信区间
    ax.fill_between(pred_data['year'], pred_data['reg_lower'], pred_data['reg_upper'], color='#ff7f0e', alpha=0.15, label='95% Prediction Interval', zorder=2)

    # 在 2030 和 2035 绘制定位十字虚线
    x_min = 2019
    for yr in [2030, 2035]:
        if yr in pred_data['year'].values:
            val = pred_data[pred_data['year'] == yr]['reg_mean'].values[0]
            # 垂直落向 X 轴
            ax.plot([yr, yr], [0, val], color='gray', linestyle=':', linewidth=1.5, zorder=1)
            # 水平落向 Y 轴
            ax.plot([x_min, yr], [val, val], color='gray', linestyle=':', linewidth=1.5, zorder=1)

    ax.set_title(f'EV Penetration S-Curve Projection: {country}', fontweight='bold', pad=15)
    ax.set_xlabel('Year', fontweight='bold', labelpad=10)
    ax.set_ylabel('EV Penetration Rate (%)', fontweight='bold', labelpad=10)

    ax.set_xlim(2019, 2035)
    ax.set_ylim(0, 60)
    ax.set_xticks(np.arange(2019, 2036, 2))
    ax.set_yticks(np.arange(0, 61, 10))

    sns.despine()
    ax.legend(frameon=False, loc='upper left')
    plt.tight_layout()

    # 导出
    filename = f"Stage4_Country_{country.replace(' ', '_')}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"生成完毕: {filename}")

# =========================================================
# 4. 绘制 12 Region聚合图
fig, ax = plt.subplots(figsize=(10, 6))
colors = sns.color_palette("tab20", n_colors=12)
regions = sorted(region_df['Region'].unique())

for i, region in enumerate(regions):
    data = region_df[region_df['Region'] == region]

    ax.plot(data['year'], data['reg_mean'], label=region, linewidth=1.5, color=colors[i], alpha=0.9)

ax.axvline(x=2025, color='black', linestyle='--', linewidth=1.5, alpha=0.6)
ax.text(2025.2, 5, 'Prediction\nStarts', color='black', fontsize=10, verticalalignment='bottom')
# ax.axhline(y=50.24, color='red', linestyle=':', linewidth=1.5, alpha=0.6)
# ax.text(2019.2, 51.5, '50.24% Maximum Mathematical Limit', color='red', fontsize=10)

ax.set_title('EV Penetration Projections by Region (12 Regions)', fontweight='bold', pad=15)
ax.set_xlabel('Year', fontweight='bold', labelpad=10)
ax.set_ylabel('EV Penetration Rate (%)', fontweight='bold', labelpad=10)

ax.set_xlim(2019, 2035)
ax.set_ylim(0, 60)
ax.set_xticks(np.arange(2019, 2036, 2))
ax.set_yticks(np.arange(0, 61, 10))
sns.despine()

ax.legend(title='Macro Region', bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, title_fontproperties={'weight': 'bold'}, fontsize=10)

plt.tight_layout()
plt.savefig("Stage4_Regions_Thin.png", dpi=300, bbox_inches='tight')
plt.close()
print("生成完毕: Stage4_Regions_Thin.png")

生成完毕: Stage4_Country_England.png
生成完毕: Stage4_Country_Northern_Ireland.png
生成完毕: Stage4_Country_Scotland.png
生成完毕: Stage4_Country_Wales.png
生成完毕: Stage4_Regions_Thin.png


In [10]:
# 1. 加载数据
df = pd.read_csv('Stage4_Module1_Future_Features.csv')
df = df.sort_values(by=['ONS_code', 'year']).reset_index(drop=True)

df_train = df[(df['year'] >= 2020) & (df['year'] <= 2025)].copy()
df_train = df_train.dropna(subset=['lag1_ports_per_10k_pop', 'lag1_gdhi_per_head', 'ev_penetration', 'pop_value'])

FLEET_CEILING = 0.5024
P_actual = (df_train['ev_penetration'] / 100.0).clip(0.0001, FLEET_CEILING - 0.001)
df_train['logit_ev'] = logit(P_actual / FLEET_CEILING)

# 重构模型
formula = "logit_ev ~ year + np.log1p(lag1_ports_per_10k_pop) + np.log1p(lag1_gdhi_per_head) + np.log1p(pop_value) + C(ONS_code)"
model = smf.ols(formula=formula, data=df_train).fit()

# 提取未来 95% 误差带
df_future = df[df['year'] >= 2026].copy()
pred_results = model.get_prediction(df_future)
pi = pred_results.conf_int(obs=True, alpha=0.05)

df_future['ev_mean'] = FLEET_CEILING * expit(pred_results.predicted_mean) * 100
df_future['ev_lower'] = FLEET_CEILING * expit(pi[:, 0]) * 100
df_future['ev_upper'] = FLEET_CEILING * expit(pi[:, 1]) * 100

# 计算绝对物理车辆
df_future['veh_mean'] = (df_future['ev_mean'] / 100.0) * df_future['veh_total']
df_future['veh_lower'] = (df_future['ev_lower'] / 100.0) * df_future['veh_total']
df_future['veh_upper'] = (df_future['ev_upper'] / 100.0) * df_future['veh_total']

df_hist = df[df['year'] <= 2025].copy()
df_hist['veh_mean'] = (df_hist['ev_penetration'] / 100.0) * df_hist['veh_total']
df_hist['veh_lower'] = df_hist['veh_mean']
df_hist['veh_upper'] = df_hist['veh_mean']

cols = ['year', 'veh_total', 'veh_mean', 'veh_lower', 'veh_upper']
df_plot = pd.concat([df_hist[cols], df_future[cols]], ignore_index=True)

# 2. UK宏观除权聚合
uk_df = df_plot.groupby(['year'])[['veh_mean', 'veh_lower', 'veh_upper', 'veh_total']].sum().reset_index()
uk_df['reg_mean'] = (uk_df['veh_mean'] / uk_df['veh_total']) * 100
uk_df['reg_lower'] = (uk_df['veh_lower'] / uk_df['veh_total']) * 100
uk_df['reg_upper'] = (uk_df['veh_upper'] / uk_df['veh_total']) * 100


# 3. 绘制UK S 曲线
sns.set_theme(style="ticks", font_scale=1.2)
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'Helvetica', 'DejaVu Sans']

hist_data = uk_df[uk_df['year'] <= 2025]
pred_data = uk_df[uk_df['year'] >= 2025]

fig, ax = plt.subplots(figsize=(10, 6))

# 真实数据段 (蓝色实线 + 实心点)
ax.plot(hist_data['year'], hist_data['reg_mean'], color='#1f77b4', linewidth=2.5, marker='o', markersize=6, label='Historical Data', zorder=4)

# 预测数据段 (橙色虚线 + 实心点)
ax.plot(pred_data['year'], pred_data['reg_mean'], color='#ff7f0e', linewidth=2.5, linestyle='--', marker='o', markersize=6, label='Predicted Data', zorder=4)

# 95% 置信区间
ax.fill_between(pred_data['year'], pred_data['reg_lower'], pred_data['reg_upper'], color='#ff7f0e', alpha=0.15, label='95% Prediction Interval', zorder=2)

# 在 2030 和 2035 绘制无数字标注的定位十字虚线
x_min = 2019
for yr in [2030, 2035]:
    if yr in pred_data['year'].values:
        val = pred_data[pred_data['year'] == yr]['reg_mean'].values[0]
        # 垂直落向 X 轴
        ax.plot([yr, yr], [0, val], color='gray', linestyle=':', linewidth=1.5, zorder=1)
        # 水平落向 Y 轴
        ax.plot([x_min, yr], [val, val], color='gray', linestyle=':', linewidth=1.5, zorder=1)

ax.set_title('EV Penetration S-Curve Projection: United Kingdom', fontweight='bold', pad=15)
ax.set_xlabel('Year', fontweight='bold', labelpad=10)
ax.set_ylabel('EV Penetration Rate (%)', fontweight='bold', labelpad=10)

ax.set_xlim(2019, 2035)
ax.set_ylim(0, 60)
ax.set_xticks(np.arange(2019, 2036, 2))
ax.set_yticks(np.arange(0, 61, 10))

sns.despine()
ax.legend(frameon=False, loc='upper left')
plt.tight_layout()

filename = "Stage4_United_Kingdom.png"
plt.savefig(filename, dpi=300, bbox_inches='tight')
plt.close()

print(f"生成完毕: {filename}")

生成完毕: Stage4_United_Kingdom.png
